# Deeptech M&A Momentum: Optimal Lag Determination
 
## Phase 3, Step 3.3: Optimal Lag Determination (Signal Testing)

This notebook performs the crucial statistical analysis to find the predictive relationship between the transformed M&A Volume features (from Step 3.2) and future sector returns. We use the **Granger Causality Test** to determine the optimal M&A frequency ('1mo', '3mo', or '6mo') and the best lead-time (lag) that provides a statistically significant signal for predicting sector rotation.
 
**Test Matrix:** For each sector, we test 18 predictive combinations (3 Frequencies x 6 Lags).

---

In [5]:
# --- 1. Standard Library Imports ---
from pathlib import Path
import sys
from typing import Dict, List, Tuple

# --- 2. Third-Party Library Imports ---
import polars as pl
import pandas as pd # Required for statsmodels compatibility
from statsmodels.tsa.stattools import grangercausalitytests

In [6]:
# --- Configuration ---
MNA_INPUT_DIR = Path("../../data/processed")
RESULTS_OUTPUT_DIR = Path("../../data/outputs")
RESULTS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True) # Ensure outputs directory exists
RESULTS_OUTPUT_PATH = RESULTS_OUTPUT_DIR / "3.3_granger_causality_results.csv"

MAX_GRANGER_LAG = 6 # Max lag to test (1 to 6 months)
CRITICAL_P_VALUE = 0.05

# Predictor columns from the MASTER file generated in 3.2
SCALED_VOLUME_COLS = ["Volume_MNA_Scaled_1mo", "Volume_MNA_Scaled_3mo", "Volume_MNA_Scaled_6mo"]

# List of Tickers to analyze (excluding the benchmark ^GSPC)
SECTOR_TICKERS = list({
    "HYDR", "IYZ", "TMET", "BOTZ", "ITA", "SNSR", "AIQ", "SOXX", 
    "LIT", "PRNT", "GRID", "ARKG", "KRBN", "XLB", "IYJ", "FAN"
})


### Step 1: Granger Causality Function Definition

We define the function that handles the creation of lagged features and execution of the statistical test.

In [ ]:
def run_granger_tests(df_data: pl.DataFrame) -> List[Dict]:
    """
    Performs Granger Causality tests, dynamically handling Insufficient Observations 
    and Zero-Variance errors across all 54 combinations per ticker.
    """
    # Define required lists within the function scope
    TARGET_RETURN_COLS = ["Returns_Target_1m", "Returns_Target_3m", "Returns_Target_6m"]
    SCALED_VOLUME_COLS = ["Volume_MNA_Scaled_1mo", "Volume_MNA_Scaled_3mo", "Volume_MNA_Scaled_6mo"]
    
    all_granger_results = []
    
    # Filter out the benchmark and sort
    df_data = df_data.filter(pl.col("Ticker").is_in(SECTOR_TICKERS)).sort("Ticker", "Date")
    
    for ticker in SECTOR_TICKERS:
        df_sector = df_data.filter(pl.col("Ticker") == ticker)
        
        # --- START TARGET LOOP (3 Horizons) ---
        for target_col in TARGET_RETURN_COLS:
            
            # --- START PREDICTOR LOOP (3 Frequencies) ---
            for volume_col in SCALED_VOLUME_COLS:
                
                # 1. Create lagged features up to MAX_GRANGER_LAG (6)
                df_test = df_sector.select(["Date", target_col, volume_col]).sort("Date")
                
                for lag in range(1, MAX_GRANGER_LAG + 1):
                    lagged_col = f"{volume_col}_Lag{lag}"
                    df_test = df_test.with_columns(
                        pl.col(volume_col).shift(lag).alias(lagged_col)
                    )

                # Prepare Pandas DataFrame for statsmodels
                # Select up to MAX_GRANGER_LAG (which is 6)
                cols_to_select = [target_col] + [f"{volume_col}_Lag{lag}" for lag in range(1, MAX_GRANGER_LAG + 1)]
                df_test_pd = df_test.select(cols_to_select).to_pandas().dropna()
                
                N_obs = len(df_test_pd)
                
                # --- FIX 1: DYNAMIC MAX LAG CALCULATION (Resolves 'Insufficient observations') ---
                # Safe Max Lag L = floor((N_obs - 2) / 2) - 1. We start the test from 1 up to this L.
                # We also cap this by the initial request (MAX_GRANGER_LAG).
                max_test_lag = min(MAX_GRANGER_LAG, max(1, (N_obs - 2) // 2 - 1))
                
                if max_test_lag < 1:
                    print(f"  -> SKIP {volume_col} vs {target_col}: Insufficient clean data ({N_obs} obs) for lag 1 test.")
                    continue
                
                # --- FIX 2: ZERO VARIANCE CHECK (Resolves 'wrong shape for coefs') ---
                predictor_vars = df_test_pd.iloc[:, 1:]
                # Check variance of the slice used for the current max_test_lag
                if (predictor_vars.var().min() < 1e-10):
                    print(f"  -> SKIP {volume_col} vs {target_col}: Predictor variance is near-zero (Constant data).")
                    continue
                # ------------------------------------------------------------------------------------
                
                print(f"  -> Testing {volume_col} vs {target_col} (Max Lag: {max_test_lag})")

                # 2. Run Granger Causality Test (Only test up to the safe max_test_lag)
                try:
                    gc_test = grangercausalitytests(
                        df_test_pd.iloc[:, :max_test_lag + 1], # Only pass columns needed for the max_test_lag
                        maxlag=max_test_lag, 
                        verbose=False
                    )
                except Exception as e:
                    print(f"Error running GC for {ticker} ({volume_col} vs {target_col}): {e}")
                    continue

                # 3. Extract results for each individual lag up to the max safe lag
                for lag in range(1, max_test_lag + 1):
                    p_value = gc_test[lag][0]['ssr_ftest'][1]
                    is_causal = p_value <= CRITICAL_P_VALUE
                    
                    all_granger_results.append({
                        'Ticker': ticker,
                        'Predictor_Freq': volume_col.split('_')[-1],
                        'Target_Horizon': target_col.split('_')[-1],
                        'Lag_Months': lag,
                        'F_Test_PValue': p_value,
                        'Causal_Relation': is_causal
                    })
                    if is_causal:
                         print(f"  -> {volume_col} | Lag {lag} -> {target_col}: {p_value:.4f} (CAUSAL)")

    return all_granger_results
                # 1. Create lagged features
                df_test = df_sector.select(["Date", target_col, volume_col]).sort("Date")
                
                for lag in range(1, max_test_lag + 1): # Use max_test_lag
                    lagged_col = f"{volume_col}_Lag{lag}"
                    df_test = df_test.with_columns(
                        pl.col(volume_col).shift(lag).alias(lagged_col)
                    )

                # Prepare Pandas DataFrame for statsmodels
                cols_to_select = [target_col] + [f"{volume_col}_Lag{lag}" for lag in range(1, max_test_lag + 1)]
                df_test_pd = df_test.select(cols_to_select).to_pandas().dropna()
                
                # --- FIX 1: ZERO VARIANCE CHECK (Resolves 'wrong shape for coefs') ---
                predictor_vars = df_test_pd.iloc[:, 1:]
                if (predictor_vars.var().min() < 1e-10):
                    print(f"  -> SKIP {volume_col} vs {target_col}: Predictor variance is near-zero.")
                    continue
                # ----------------------------------------------------------------------
                
                # 2. Run Granger Causality Test
                try:
                    gc_test = grangercausalitytests(
                        df_test_pd, 
                        maxlag=max_test_lag, 
                        verbose=False
                    )
                except Exception as e:
                    # FIX 2: Dynamic lag reduction handled by max_test_lag, this captures any final edge cases
                    print(f"Error running GC for {ticker} ({volume_col} vs {target_col}): {e}")
                    continue

                # 3. Extract results for each individual lag
                for lag in range(1, max_test_lag + 1): # Use max_test_lag
                    p_value = gc_test[lag][0]['ssr_ftest'][1]
                    is_causal = p_value <= CRITICAL_P_VALUE
                    
                    all_granger_results.append({
                        'Ticker': ticker,
                        'Predictor_Freq': volume_col.split('_')[-1],
                        'Target_Horizon': target_col.split('_')[-1],
                        'Lag_Months': lag,
                        'F_Test_PValue': p_value,
                        'Causal_Relation': is_causal
                    })
                    if is_causal:
                         print(f"  -> {volume_col} | Lag {lag} -> {target_col}: {p_value:.4f} (CAUSAL)")

    return all_granger_results

### Step 2: Execution and Saving Results

We load the single master feature file and execute the tests, saving the results to `data/outputs/`.

In [8]:
# --- Execution ---

INPUT_PATH = MNA_INPUT_DIR / "3.2_aligned_features_MASTER.csv"
    
if not INPUT_PATH.exists():
    print(f"ERROR: Master Feature CSV not found at {INPUT_PATH}. Please run 3.2 first.")
else:
    # Load the master file
    df_master = pl.read_csv(INPUT_PATH)
    
    # Run the tests
    results = run_granger_tests(df_master)
    
    # Save the results
    df_granger_results = pd.DataFrame(results)
    df_granger_results.to_csv(RESULTS_OUTPUT_PATH, index=False)
    
    print("\n" + "=" * 80)
    print(f"✓ All Granger Causality tests complete.")
    print(f"Results saved to {RESULTS_OUTPUT_PATH}")
    print("=" * 80)


--- Testing Ticker: KRBN (Max Lag: 6) ---
Error running GC for KRBN (Volume_MNA_Scaled_1mo vs Returns_Target_1m): wrong shape for coefs
Error running GC for KRBN (Volume_MNA_Scaled_3mo vs Returns_Target_1m): wrong shape for coefs
Error running GC for KRBN (Volume_MNA_Scaled_6mo vs Returns_Target_1m): wrong shape for coefs
Error running GC for KRBN (Volume_MNA_Scaled_1mo vs Returns_Target_3m): wrong shape for coefs
Error running GC for KRBN (Volume_MNA_Scaled_3mo vs Returns_Target_3m): wrong shape for coefs
Error running GC for KRBN (Volume_MNA_Scaled_6mo vs Returns_Target_3m): wrong shape for coefs
Error running GC for KRBN (Volume_MNA_Scaled_1mo vs Returns_Target_6m): wrong shape for coefs
Error running GC for KRBN (Volume_MNA_Scaled_3mo vs Returns_Target_6m): wrong shape for coefs
Error running GC for KRBN (Volume_MNA_Scaled_6mo vs Returns_Target_6m): wrong shape for coefs

--- Testing Ticker: FAN (Max Lag: 6) ---
Error running GC for FAN (Volume_MNA_Scaled_1mo vs Returns_Target_1m)

c:\Users\mathi\Documents\Code\deeptech-ma-momentum\.venv\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
c:\Users\mathi\Documents\Code\deeptech-ma-momentum\.venv\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
c:\Users\mathi\Documents\Code\deeptech-ma-momentum\.venv\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
c:\Users\mathi\Documents\Code\deeptech-ma-momentum\.venv\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
c:\Users\mathi\Documents\Code\deeptech-ma-momentum\.venv\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print 

Error running GC for IYZ (Volume_MNA_Scaled_6mo vs Returns_Target_6m): wrong shape for coefs

--- Testing Ticker: PRNT (Max Lag: 6) ---
Error running GC for PRNT (Volume_MNA_Scaled_1mo vs Returns_Target_1m): wrong shape for coefs
Error running GC for PRNT (Volume_MNA_Scaled_3mo vs Returns_Target_1m): wrong shape for coefs
Error running GC for PRNT (Volume_MNA_Scaled_6mo vs Returns_Target_1m): wrong shape for coefs
Error running GC for PRNT (Volume_MNA_Scaled_1mo vs Returns_Target_3m): wrong shape for coefs
Error running GC for PRNT (Volume_MNA_Scaled_3mo vs Returns_Target_3m): wrong shape for coefs
Error running GC for PRNT (Volume_MNA_Scaled_6mo vs Returns_Target_3m): wrong shape for coefs
Error running GC for PRNT (Volume_MNA_Scaled_1mo vs Returns_Target_6m): wrong shape for coefs
Error running GC for PRNT (Volume_MNA_Scaled_3mo vs Returns_Target_6m): wrong shape for coefs
Error running GC for PRNT (Volume_MNA_Scaled_6mo vs Returns_Target_6m): wrong shape for coefs

--- Testing Ticke

c:\Users\mathi\Documents\Code\deeptech-ma-momentum\.venv\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
c:\Users\mathi\Documents\Code\deeptech-ma-momentum\.venv\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
c:\Users\mathi\Documents\Code\deeptech-ma-momentum\.venv\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
c:\Users\mathi\Documents\Code\deeptech-ma-momentum\.venv\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
c:\Users\mathi\Documents\Code\deeptech-ma-momentum\.venv\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print 

Error running GC for IYJ (Volume_MNA_Scaled_6mo vs Returns_Target_6m): wrong shape for coefs

--- Testing Ticker: ITA (Max Lag: 6) ---
Error running GC for ITA (Volume_MNA_Scaled_1mo vs Returns_Target_1m): wrong shape for coefs
Error running GC for ITA (Volume_MNA_Scaled_3mo vs Returns_Target_1m): wrong shape for coefs
Error running GC for ITA (Volume_MNA_Scaled_6mo vs Returns_Target_1m): wrong shape for coefs
Error running GC for ITA (Volume_MNA_Scaled_1mo vs Returns_Target_3m): wrong shape for coefs
Error running GC for ITA (Volume_MNA_Scaled_3mo vs Returns_Target_3m): wrong shape for coefs
Error running GC for ITA (Volume_MNA_Scaled_6mo vs Returns_Target_3m): wrong shape for coefs
Error running GC for ITA (Volume_MNA_Scaled_1mo vs Returns_Target_6m): wrong shape for coefs
Error running GC for ITA (Volume_MNA_Scaled_3mo vs Returns_Target_6m): wrong shape for coefs
Error running GC for ITA (Volume_MNA_Scaled_6mo vs Returns_Target_6m): wrong shape for coefs

✓ All Granger Causality tes

c:\Users\mathi\Documents\Code\deeptech-ma-momentum\.venv\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
c:\Users\mathi\Documents\Code\deeptech-ma-momentum\.venv\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
c:\Users\mathi\Documents\Code\deeptech-ma-momentum\.venv\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
c:\Users\mathi\Documents\Code\deeptech-ma-momentum\.venv\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(
c:\Users\mathi\Documents\Code\deeptech-ma-momentum\.venv\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print 